# Validating DC OPF Model, April 14, 2025

In [11]:
import pandas as pd
import pandapower as pp
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import copy
import copy
import pulp
import time
import pulp
import csv
import random

#MY CODE BASE
import GridFlowSmooth as flow 

datelabel = str(datetime.now())[2:10]

def ACDC_TEP_OPF(gen_capacity = {1:100,2:100,3:0,4:0,5:100, 6:0, 7:0}, line_capacity = 1000, 
                 susceptances = {(1,2):.1, (4,6):.1, (5,7):.1, (7,2):.1, (1,3):.1, (2,3):.1, (2,4):.1, (3,5):.1}, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4, 5, 6, 7], obj = 'curtail',
                 demands = {1: 50, 2: 60, 3: 40, 4: 30, 5:50, 6:20, 7:60}, cut = False,
                 ac_lines = [(1, 2), (4,6), (5,7), (7,2)], poss_ac_lines = [(1, 3),(2, 3), (2, 4), (3,5)],
                 dc_lines = [(3,4)], poss_dc_lines = [(3,4)], generation_cost = {1: 120, 2: 25, 5: 10},
                 printout = False, max_new_ac = 1, max_new_dc = 0, label=datelabel, save = True):
    #saving input
    M = 654321
    n = len(buses)
    if save:
        busM = np.zeros((n,3)) #  bus id, demand, gen_capacity
        k = 0
        for i in buses:
            busM[k, 0] = i
            busM[k, 1] = demands[i]
            busM[k, 2] = gen_capacity[i]
            k+=1
        # saving node_from, node_to, sus, capacity to later rebuild adjacency matrix 
        # AC, new AC, old DC, then new DC. separated by rows of -1s

        old_ac = np.array([(ac[0], ac[1], susceptances[ac], line_capacity) for ac in ac_lines])
        new_ac = np.array([(ac[0], ac[1], susceptances[ac], line_capacity) for ac in poss_ac_lines])
        old_dc = np.array([(dc[0], dc[1],0, line_capacity) for dc in dc_lines])
        new_dc = np.array([(dc[0], dc[1],0, line_capacity) for dc in poss_dc_lines])
        pause =- np.array([[1,1,1,1]])
        if dc_lines == []: # concatinating the empty array of dc lines is an issue
            connections = np.concatenate([old_ac, pause, new_ac, pause, new_dc], axis=0)
        else:
            connections = np.concatenate([old_ac, pause, new_ac, pause, old_dc, pause, new_dc], axis=0)
            
        np.save('GNN_Data/busM'+label+'.npy', busM)
        np.save('GNN_Data/linesM'+label+'.npy', connections )
 
    prob = pulp.LpProblem("Transmission_Expansion_DC_OPF", pulp.LpMinimize)    
    # Decision Variables
    # Generation at each bus
    generators = gen_capacity.keys()
    gen = {b: pulp.LpVariable(f"gen_{b}", 0, gen_capacity[b]) for b in generators}
    gen.update({b: pulp.LpVariable(f"gen_{b}", 0, 0) for b in buses if b not in generators})
    
    #curtailment
    if cut:
        curtail = {b: pulp.LpVariable(f"curtail_{b}", 0, demand[b] ) for b in buses}
    else:
        curtail = {b: pulp.LpVariable(f"curtail_{b}", 0, 0) for b in buses}
               
    # Binary variables for new lines
    new_line = {(i, j): pulp.LpVariable(f"new_line_{i}_{j}", cat="Binary")
                for i, j in poss_ac_lines}
    new_dc_line = {(i,j): pulp.LpVariable(f"new_dc_line_{i}_{j}", cat= "Binary") 
                   for i, j in poss_dc_lines}
    
    # Power flow on lines
    flow = {(i, j): pulp.LpVariable(f"flow_{i}_{j}", -line_capacity, line_capacity) 
            for i, j in ac_lines + poss_ac_lines}
    dc_flow = {(i,j): pulp.LpVariable(f"DC_flow_{i}_{j}", -line_capacity, line_capacity) 
               for i, j in dc_lines + poss_dc_lines}

    # Voltage phase at each bus (reference phase at bus 1 is 0)
    phase = {b: pulp.LpVariable(f"phase_{b}", -np.pi/6, np.pi/6) for b in buses}
    if obj == 'curtail':
        prob += (pulp.lpSum(curtail[b] for b in buses)) 
        
        # MINIMIZE CUT POWER
    elif obj == 'cost':
        prob += (pulp.lpSum(gen[b] * generation_cost[b] for b in generation_cost) )
        # MINIMIZE COST OF FULFILLMENT
    else:
        raise ValueError(f' obj variable received {obj} rather than `cost` or `curtail`')
    # Constraints
    # 1. Power balance at each bus
    for b in buses:
        prob += (pulp.lpSum(flow[i, b] for i, _ in flow if b == _) 
                 - pulp.lpSum(flow[b, j] for _, j in flow if b == _) 
                 + pulp.lpSum(dc_flow[i, b] for i, _ in dc_flow if b== _)
                 - pulp.lpSum(dc_flow[b, j] for _, j in dc_flow if b== _)
                 + gen[b] + curtail[b] == demands[b], f"Power_Balance_{b}")
    
    # 2. Enforce line flow limits for candidate lines only when they are added
    for i, j in poss_ac_lines:
        prob += (flow[i, j] <= line_capacity*new_line[i, j] , f"Line_Capacity_Pos_{i}_{j}")
        prob += (flow[i, j] >= -line_capacity* new_line[i, j] , f"Line_Capacity_Neg_{i}_{j}")
    

    for i, j in poss_dc_lines:
        prob += (dc_flow[i, j] <= line_capacity* new_dc_line[i, j], f"DC_Line_Capacity_Pos_{i}_{j}")
        prob += (dc_flow[i, j] >= -line_capacity *new_dc_line[i, j], f"DC_Line_Capacity_Neg_{i}_{j}")
        
    # 3. Flow constraints based on phase differences and susceptance
    for i, j in ac_lines: #+ poss_ac_lines:
        prob += (flow[i, j] == susceptances[(i,j)] * (phase[i] - phase[j]), f"Flow_phase_{i}_{j}")

    for i, j in poss_ac_lines:
        prob += (susceptances[(i,j)] * (phase[i] - phase[j]) - flow[i, j] <= M*(1- new_line[i, j]) , f"Line_Flow_Pos_{i}_{j}")
        prob += (susceptances[(i,j)] * (phase[i] - phase[j]) - flow[i, j] >= -M*(1-  new_line[i, j]) , f"Line_Flow_Neg_{i}_{j}")
    
    
    # but DC flows freely :) as the breathed wind o'er the lush forests
    slack_gen = np.min(list(generation_cost.keys()))
    # 4. Set reference bus phase to 0
    prob += (phase[slack_gen] ==0 , "Reference_Bus_phase")
    prob += (pulp.lpSum(new_line[(i,j)] for i,j in poss_ac_lines) <= max_new_ac, 'max_AC_line')
    prob += (pulp.lpSum(new_dc_line[(i,j)] for i,j in poss_dc_lines) <= max_new_dc, 'max_DC_line')
    # Solve the problem
    prob.solve(pulp.GUROBI_CMD())


    #save the output, generation, curtailment and new lines
    if save:
        busY = np.zeros((n,3)) #  buses' curtailment, generation
        k = 0
        for i in buses:
            busY[k,0] = i
            busY[k,1] = curtail[i].varValue
            busY[k,2] = gen[i].varValue
            k+=1
        #output records matrix for new AC, then new DC
        #each row has from_node, to_node, susceptance, capacity, added. 
        #separated by rows of -1s
        added_ac = [(i,j, susceptances[i,j], line_capacity, new_line[i,j].varValue) for (i,j) in new_line]
        added_dc = [(i,j, 0, line_capacity, new_dc_line[i,j].varValue) for (i,j) in new_dc_line]
        pause =- np.array([[1,1,1,1,1]])
        new_connections = np.concatenate([added_ac, pause, added_dc], axis=0)
        
        np.save('GNN_Data/busY'+label+'.npy', busY)
        np.save('GNN_Data/linesNewY'+label+'.npy', new_connections)
    
    # Print results
    if printout:
        print("Status:", pulp.LpStatus[prob.status])
        print("Objective Value:", value(prob.objective))
        print("\nGeneration at Each Bus:")
        for b in buses:
            if gen[b].varValue>0:
                print(f"Bus {b}: {gen[b].varValue} MW")
        if cut:
            print("\nCurtailment:")
            for b in buses:
                if curtail[b].varValue>0:
                    print(f"Bus {b}: {curtail[b].varValue} MW")
        print("\nAC Line Flows:")
        for i, j in ac_lines + poss_ac_lines:
            # if flow[i, j].varValue >0:
            print(f"Line {i}-{j}: {flow[i, j].varValue} MW")
        if dc_lines != []:
            print("\nDC Line Flows:")
            for i, j in dc_lines + poss_dc_lines:
                print(f"Line {i}-{j}: {dc_flow[i, j].varValue} MW")
        if max_new_dc + max_new_ac != 0:
            print("\nNew AC Line Decisions:")
            for i, j in poss_ac_lines:
                print(f"Line {i}-{j}: {'Added' if new_line[i, j].varValue > 0.5 else 'Not Added'}")
            print("\nNew DC Line Decisions:")
            for i, j in poss_dc_lines:
                print(f"Line {i}-{j}: {'Added' if new_dc_line[i, j].varValue > 0.5 else 'Not Added'}")
        print("\nVoltage phases:")
        for b in buses:
            print(f"Bus {b}: {phase[b].varValue} rads")
    all_out = {'gen': {i:gen[i].varValue for i in gen}, 'curtail':{i:curtail[i].varValue for i in curtail},
        'ac_lines':ac_lines, 'dc_lines':dc_lines,  'phase':{i:phase[i].varValue for i in phase}, 
        'possible_ac':poss_ac_lines, 'possible_dc':poss_dc_lines,
       'ac_flow':{i:flow[i].varValue for i in flow}, 'dc_flow':{i:dc_flow[i].varValue for i in dc_flow},
       'new_line':{i:new_line[i].varValue for i in new_line}, 
        'new_dc_line': {i:new_dc_line[i].varValue for i in new_dc_line} }
    return all_out, prob

In [7]:
!python --version

Python 3.12.8


In [13]:
!pip list

Package                   Version
------------------------- -----------
absl-py                   2.1.0
apache-beam               2.63.0
asttokens                 3.0.0
astunparse                1.6.3
attrs                     25.1.0
Bottleneck                1.4.2
branca                    0.7.2
Brotli                    1.1.0
certifi                   2025.1.31
cffi                      1.17.1
cftime                    1.6.4
charset-normalizer        3.4.0
click                     8.1.7
cloudpickle               2.2.1
colorama                  0.4.6
colorlover                0.3.0
comm                      0.2.2
contourpy                 1.3.1
crcmod                    1.7
cufflinks                 0.17.3
cycler                    0.12.1
dask                      2024.12.0
decorator                 5.1.1
deprecation               2.1.0
dill                      0.3.1.1
dnspython                 2.7.0
docopt                    0.6.2
executing                 2.2.0
fastavro           

In [15]:
from gurobipy import GRB

ModuleNotFoundError: No module named 'gurobipy'

In [19]:
import sys
print(sys.path)

['C:\\Users\\DJFRO\\Desktop\\Todo\\Hubs\\TransmissionDelay', 'C:\\Users\\DJFRO\\anaconda3\\python312.zip', 'C:\\Users\\DJFRO\\anaconda3\\DLLs', 'C:\\Users\\DJFRO\\anaconda3\\Lib', 'C:\\Users\\DJFRO\\anaconda3', '', 'C:\\Users\\DJFRO\\anaconda3\\Lib\\site-packages', 'C:\\Users\\DJFRO\\anaconda3\\Lib\\site-packages\\win32', 'C:\\Users\\DJFRO\\anaconda3\\Lib\\site-packages\\win32\\lib', 'C:\\Users\\DJFRO\\anaconda3\\Lib\\site-packages\\Pythonwin', 'C:\\Users\\DJFRO\\anaconda3\\Lib\\site-packages\\setuptools\\_vendor']


In [21]:
!jupyter --paths

config:
    C:\Users\DJFRO\.jupyter
    C:\Users\DJFRO\AppData\Roaming\Python\etc\jupyter
    C:\Users\DJFRO\anaconda3\etc\jupyter
    C:\ProgramData\jupyter
data:
    C:\Users\DJFRO\AppData\Roaming\jupyter
    C:\Users\DJFRO\AppData\Roaming\Python\share\jupyter
    C:\Users\DJFRO\anaconda3\share\jupyter
    C:\ProgramData\jupyter
runtime:
    C:\Users\DJFRO\AppData\Roaming\jupyter\runtime


In [3]:


options = {
    "WLSACCESSID": "access-id",
    "WLSSECRET": "secret",
    "LICENSEID": "license-id",
}

with gp.Env(params=options) as env:
    # Pass environment as a parameter
    solver = pulp.GUROBI(env=env)
    prob.solve(solver)
    solver.close()

ModuleNotFoundError: No module named 'gurobipy'

In [15]:
ACDC_TEP_OPF(gen_capacity = {1:100,2:100,3:0,4:0,5:100, 6:0, 7:0}, line_capacity = 1000, 
                 susceptances = {(1,2):.1, (4,6):.1, (5,7):.1, (7,2):.1, (1,3):.1, (2,3):.1, (2,4):.1, (3,5):.1}, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4, 5, 6, 7], obj = 'curtail',
                 demands = {1: 50, 2: 60, 3: 40, 4: 30, 5:50, 6:20, 7:60}, cut = False,
                 ac_lines = [(1, 2), (4,6), (5,7), (7,2)], poss_ac_lines = [(1, 3),(2, 3), (2, 4), (3,5)],
                 dc_lines = [(3,4)], poss_dc_lines = [(3,4)], generation_cost = {1: 120, 2: 25, 5: 10},
                 printout = False, max_new_ac = 1, max_new_dc = 0, label=datelabel, save = False)

PulpSolverError: PuLP: Error while trying to execute gurobi_cl.exe

# PandaPower Solver as Reference
"joint development of the research group of the Department for Sustainable Electrical Energy Systems (e2n), University of Kassel and the Department for Distribution System Operation at the Fraunhofer Institute for Energy Economics and Energy System Technology (IEE), Kassel." 
https://pandapower.readthedocs.io/en/latest/


In [4]:
def panda_DCOPF(gen_capacity = {1:150,2:0,3:0,4:0}, line_capacity = 100, 
                 susceptances = {(1,2):1500, (2,3):1500, (3,4):1500, (1,3):1500, (1,4):1500}, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4], obj = 'cost', cut = False,
                 demands = {1: 30, 2: 20, 3: 20, 4: 20}, ac_lines = [(1, 2), (2,3), (3,4)], 
                 dc_lines = [], generation_cost = {1: 120}, printout = True):
    net = pp.create_empty_network()
    ## # BUSES # ##
    pd_buses = {}             
    for i in buses:
        pd_buses[i] = pp.create_bus(net, vn_kv=1, name=f"Bus {i}")
    ## # LINES # ##
    for fb, tb in ac_lines:
        pp.create_line_from_parameters(net, from_bus=pd_buses[fb], to_bus=pd_buses[tb], length_km=1,
                                       r_ohm_per_km=0, x_ohm_per_km=1/susceptances[(fb,tb)], c_nf_per_km=0,
                                       max_i_ka=1000, name=f"Line {fb}-{tb}")
    ## # LOADS # ##
    for i in demands:
        pp.create_load(net, bus=pd_buses[i], p_mw=demands[i], q_mvar=0, controllable=False, name=f"Load Bus {i}")
    pd_gens = {}
    ## # GENERATORS # ##
    slack_gen = np.min(list(generation_cost.keys()))
    for i in gen_capacity:
        if gen_capacity[i]>0:
            pd_gens[i] = pp.create_gen(net, bus=pd_buses[i], p_mw=0, vm_pu=1.02, min_p_mw=0, max_p_mw=gen_capacity[i],
                         controllable=True, name=f"Gen Bus {i}", slack = i==slack_gen )
    for i in pd_gens:
        pp.create_poly_cost(net, element=pd_gens[i], et="gen", cp1_eur_per_mw=generation_cost[i])
    # Run DC OPF
    out =pp.runopp(net, delta=1e-10, calculate_voltage_angles=True, trafo_model="pi")
    if printout:
        print("\n--- Generator Results ---")
        print(net.res_gen['p_mw'])
        
        print("\n--- Line Flow ---")
        print(net.res_line[["p_from_mw"]])
        
        print("\n--- Bus Voltage Angles (rad) ---")
        print(net.res_bus["va_degree"]*np.pi/180)
    return net

## Here is our model, running on python pulp which will soon use gurobi

In [5]:
out, prob = flow.ACDC_TEP_OPF(gen_capacity = {1:150,2:0,3:0,4:0}, line_capacity = 100, 
                 susceptances = {(1,2):1500, (2,3):1500, (3,4):1500, (1,3):1500, (1,4):1500}, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4], obj = 'cost', cut = False,
                 demands = {1: 30, 2: 20, 3: 20, 4: 20}, ac_lines = [(1, 2), (2,3), (3,4)], 
                 poss_ac_lines = [(1, 3), (1,4)], dc_lines = [], poss_dc_lines = [(3,1)], 
                 generation_cost = {1: 120}, printout = True,
                 max_new_ac = 0, max_new_dc = 0, label=datelabel, save = False)

Status: Optimal
Objective Value: 10800.0

Generation at Each Bus:
Bus 1: 90.0 MW

AC Line Flows:
Line 1-2: 60.0 MW
Line 2-3: 40.0 MW
Line 3-4: 20.0 MW
Line 1-3: 0.0 MW
Line 1-4: 0.0 MW

Voltage phases:
Bus 1: 0.0 rads
Bus 2: -0.04 rads
Bus 3: -0.066666667 rads
Bus 4: -0.08 rads


## Our results above resemble pandapower results below

In [6]:
net = panda_DCOPF()


--- Generator Results ---
0    90.0
Name: p_mw, dtype: float64

--- Line Flow ---
   p_from_mw
0       60.0
1       40.0
2       20.0

--- Bus Voltage Angles (rad) ---
0    0.000000
1   -0.039954
2   -0.066645
3   -0.079997
Name: va_degree, dtype: float64


# Now let's run a general test

In [8]:
def suggest_rand(pairs, bus_list, n=3):
    out = []
    for i in range(n):
        a,b = random.choice(bus_list),random.choice(bus_list)
        while (a,b) in pairs+out or (b,a) in pairs+out:
            a = random.choice(bus_list)
            b = random.choice(bus_list)
        out.append((a,b))
    return out

This will loop as follows:
* generate a random network of variable node count, demands, generators, and generation capacity.
* run the Pandapower DC OPF solver
* run our pulp solver
* compare results
* if infeasible, each reports separately. Ideally they run or fall together

In [135]:
for i in range(15):
    print(f"\n## # Running Simulated Grid Number {i+1} # ##\n"*3)
    buses = np.arange(np.random.randint(6,13))
    n = len(buses)
    demands = {bus:np.random.randint(3,7)*30 for bus in buses}
    line_capacity = 1000
    line_expansion_cost = 1000
    obj = 'cost'
    cut = False
    ac_lines = suggest_rand([], buses,
                            n = np.random.randint(len(buses)+4, 2*len(buses)))
    poss_ac_lines = suggest_rand(ac_lines, buses, n = 1)
    poss_dc_lines = suggest_rand([], buses, n = 1)
    dc_lines = []
    susceptances = {line : np.random.randint(10,20)*100 for line in ac_lines + poss_ac_lines}  

    gen_buses = np.random.choice(buses, np.random.randint(n//5, n//3), replace = False)
    gen_capacity = {g : np.random.randint(3,7)*250* 
                           int(g in gen_buses) for g in buses}
    generation_cost = {g : np.random.randint(1,100) for g in buses if g in gen_buses}
    printout = True
    max_new_ac = 0 
    max_new_dc = 0
    label=datelabel
    save = False
    print('Total Demand:',np.sum(list(demands.values())))
    print('Gen Costs:', generation_cost)
    
    try:
        net = panda_DCOPF(gen_capacity = gen_capacity, line_capacity = line_capacity, 
                     susceptances = susceptances, 
                     line_expansion_cost = line_expansion_cost, buses = buses, obj = 'cost', cut = False,
                     demands = demands, ac_lines = ac_lines, 
                     dc_lines = dc_lines,
                     generation_cost = generation_cost, printout = False)
        print('Panda Converged')
    except Exception:
        print('Panda Infeasible')
    
    out, prob = flow.ACDC_TEP_OPF(gen_capacity = gen_capacity, line_capacity = line_capacity, 
                     susceptances = susceptances, 
                     line_expansion_cost = line_expansion_cost, buses = buses, obj = 'cost', cut = False,
                     demands = demands, ac_lines = ac_lines, 
                     poss_ac_lines = poss_ac_lines, dc_lines = dc_lines, poss_dc_lines = poss_dc_lines, 
                     generation_cost = generation_cost, printout = False,
                     max_new_ac = 0, max_new_dc = 0, label=datelabel, save = False)
    if prob.status ==-1:
        print('Our Model Infeasible\n')
    else:
        print('Our Model Converged\n')
        flow_comp = pd.DataFrame({'PandaPower MW':list(net.res_line['p_from_mw'].values), 
                              'OurModel MW':list(out['ac_flow'].values())[:-1]})
        flow_comp['Error MW'] = flow_comp['PandaPower MW']-flow_comp['OurModel MW']
        phase_comp = pd.DataFrame({'PandaPower rads':list(net.res_bus['va_degree'].values*np.pi/180), 
                                  'OurModel rads':list(out['phase'].values())})
        phase_comp['Error rads'] = phase_comp['PandaPower rads']-phase_comp['OurModel rads']
        print("Our generators:")
        print([out['gen'][key] for key in out['gen'] if out['gen'][key] > 0])
        print('Panda generators:')
        print(net.res_gen[['p_mw']])
        print("Line Flows:\n", flow_comp)
        print("Bus Phases:\n", phase_comp)



## # Running Simulated Grid Number 1 # ##

## # Running Simulated Grid Number 1 # ##

## # Running Simulated Grid Number 1 # ##

Total Demand: 1350
Gen Costs: {1: 7, 3: 75}
Panda Converged
Our Model Converged

Our generators:
[750.0, 600.0]
Panda generators:
         p_mw
0  749.999961
1  600.000039
Line Flows:
     PandaPower MW  OurModel MW      Error MW
0     -150.000000  -150.000000  2.684146e-10
1       12.645992    12.593057  5.293460e-02
2      141.058745   140.733930  3.248154e-01
3      -50.596147   -50.840086  2.439391e-01
4      -71.806182   -71.825472  1.929002e-02
5      -31.983180   -32.194541  2.113608e-01
6       70.502358    70.776162 -2.738039e-01
7     -196.028792  -195.907200 -1.215924e-01
8       -8.941255    -9.266071  3.248162e-01
9       50.510172    50.693315 -1.831429e-01
10    -200.176998  -200.369070  1.920718e-01
11     128.750473   128.347790  4.026834e-01
12     163.926734   163.883770  4.296365e-02
13     -36.002594   -35.969852 -3.274168e-02
14     102